# 06 Condition Classification (Bonus)

This notebook explores whether vehicle condition can be predicted from other attributes. While the primary focus of this project is price prediction (notebooks 03-04), condition classification is an interesting complementary question: can we estimate a car's condition rating using only its year, mileage, and market value?

We test three approaches:
1. **KNN** (multi-class): classify condition as Poor, Fair, or Good
2. **KNN** (binary): classify as Good vs Not Good
3. **Logistic Regression** (multi-class): compare a parametric model against KNN

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## Data Preparation

We use three numeric features (`year`, `odometer`, `mmr`) and bin the continuous `condition` column into three categories:
- **Poor**: 0 - 2
- **Fair**: 2 - 3
- **Good**: 3 - 5

In [2]:
car_df = pd.read_csv('data/raw/car_prices.csv')
car_df = car_df.dropna(subset=['year', 'odometer', 'mmr', 'condition'])

X = car_df[['year', 'odometer', 'mmr']]
y = pd.cut(car_df['condition'], bins=[0, 2, 3, 5], labels=['Poor', 'Fair', 'Good'])

print(f"Class distribution:")
print(y.value_counts().sort_index())
print(f"\nClass proportions:")
print((y.value_counts(normalize=True).sort_index() * 100).round(1))

Class distribution:
condition
Poor     71707
Fair    122495
Good    352786
Name: count, dtype: int64

Class proportions:
condition
Poor    13.1
Fair    22.4
Good    64.5
Name: proportion, dtype: float64


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]:,} samples")
print(f"Test:     {X_test.shape[0]:,} samples")

Training: 437,590 samples
Test:     109,398 samples


## KNN Classification (Multi-class)

We test two values of k to observe the effect on overfitting.

In [4]:
results = []

for k in [5, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    train_acc = accuracy_score(y_train, knn.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, knn.predict(X_test_scaled))
    
    results.append({'Model': f'KNN (k={k})', 'Train Acc': round(train_acc, 4), 'Test Acc': round(test_acc, 4)})
    
    print(f"\nKNN (k={k}):")
    print(f"  Train accuracy: {train_acc:.4f}")
    print(f"  Test accuracy:  {test_acc:.4f}")
    print(f"\n{classification_report(y_test, knn.predict(X_test_scaled))}")


KNN (k=5):
  Train accuracy: 0.7481
  Test accuracy:  0.6551

              precision    recall  f1-score   support

        Fair       0.35      0.32      0.33     24448
        Good       0.76      0.85      0.80     70618
        Poor       0.47      0.25      0.33     14332

    accuracy                           0.66    109398
   macro avg       0.53      0.48      0.49    109398
weighted avg       0.63      0.66      0.64    109398


KNN (k=11):
  Train accuracy: 0.7219
  Test accuracy:  0.6759

              precision    recall  f1-score   support

        Fair       0.38      0.28      0.32     24448
        Good       0.76      0.89      0.82     70618
        Poor       0.49      0.29      0.37     14332

    accuracy                           0.68    109398
   macro avg       0.54      0.49      0.50    109398
weighted avg       0.64      0.68      0.65    109398



Increasing k from 5 to 11 reduces overfitting (smaller train-test gap) but does not substantially improve test accuracy. The model performs well on the "Good" class but struggles with "Poor" and "Fair" due to class imbalance.

## KNN Classification (Binary)

Simplifying to a binary problem (Good vs Not Good) to see if the model improves when the task is easier.

In [5]:
y_binary = (y == 'Good').astype(int)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled = scaler.transform(X_test_b)

knn_binary = KNeighborsClassifier(n_neighbors=7, weights='distance')
knn_binary.fit(X_train_b_scaled, y_train_b)

train_acc = accuracy_score(y_train_b, knn_binary.predict(X_train_b_scaled))
test_acc = accuracy_score(y_test_b, knn_binary.predict(X_test_b_scaled))

results.append({'Model': 'KNN Binary (k=7)', 'Train Acc': round(train_acc, 4), 'Test Acc': round(test_acc, 4)})

print(f"KNN Binary (k=7, distance-weighted):")
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")
print(f"\n{classification_report(y_test_b, knn_binary.predict(X_test_b_scaled), target_names=['Not Good', 'Good'])}")

KNN Binary (k=7, distance-weighted):
  Train accuracy: 0.9995
  Test accuracy:  0.7223

              precision    recall  f1-score   support

    Not Good       0.62      0.56      0.59     38780
        Good       0.77      0.81      0.79     70618

    accuracy                           0.72    109398
   macro avg       0.70      0.69      0.69    109398
weighted avg       0.72      0.72      0.72    109398



The binary task achieves higher accuracy, but the large gap between train (~1.0) and test accuracy reveals significant overfitting when using distance-weighted KNN.

## Logistic Regression (Multi-class)

In [6]:
log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_train_scaled, y_train)

train_acc = accuracy_score(y_train, log_reg.predict(X_train_scaled))
test_acc = accuracy_score(y_test, log_reg.predict(X_test_scaled))

results.append({'Model': 'Logistic Regression', 'Train Acc': round(train_acc, 4), 'Test Acc': round(test_acc, 4)})

print(f"Logistic Regression:")
print(f"  Train accuracy: {train_acc:.4f}")
print(f"  Test accuracy:  {test_acc:.4f}")
print(f"\n{classification_report(y_test, log_reg.predict(X_test_scaled))}")

Logistic Regression:
  Train accuracy: 0.6918
  Test accuracy:  0.6916

              precision    recall  f1-score   support

        Fair       0.41      0.21      0.28     24448
        Good       0.75      0.94      0.83     70618
        Poor       0.52      0.32      0.39     14332

    accuracy                           0.69    109398
   macro avg       0.56      0.49      0.50    109398
weighted avg       0.64      0.69      0.65    109398



## Results Summary

In [7]:
results_df = pd.DataFrame(results).set_index('Model')
results_df['Overfit Gap'] = (results_df['Train Acc'] - results_df['Test Acc']).round(4)
results_df

,Train Acc,Test Acc,Overfit Gap
Model,,,
KNN (k=5),0.7481,0.6551,0.0930
KNN (k=11),0.7219,0.6759,0.0460
KNN Binary (k=7),0.9995,0.7223,0.2772
Logistic Regression,0.6918,0.6916,0.0002


## Conclusion

Predicting vehicle condition from year, mileage, and MMR alone proves challenging:

- **Class imbalance** is the primary issue -- "Good" dominates the dataset (~65%), so all models achieve decent accuracy by simply predicting "Good" most of the time, while struggling with "Poor" and "Fair."
- **Logistic Regression** achieves the best balance between accuracy and generalization (smallest overfit gap), despite being the simplest model.
- **KNN** overfits significantly, especially with distance weighting and small k values.

To meaningfully improve condition prediction, future work could explore:
- Oversampling minority classes (SMOTE)
- Adding more features (e.g., make, model, body type)
- Using tree-based classifiers that handle class imbalance better